# XGBoost Residual Stack Submission

Self-contained Kaggle code-competition notebook. It trains an inference-safe XGBoost base model, learns a shrunk Ridge residual correction from well-level out-of-fold base residuals, and writes `/kaggle/working/submission.csv`.

## 1. Setup

In [1]:
"Imports, constants, and robust data discovery."

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import pairwise_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

KAGGLE_ROOT = Path("/kaggle")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")

MODEL_NAME = "xgb_ridge_residual_stack_inference_safe"
MAX_TARGET_ROWS_PER_WELL = 120
RANDOM_STATE = 20260628
ANALOG_LAGS = np.arange(-40.0, 42.5, 2.5)
N_OOF_FOLDS = 3
RESIDUAL_SHRINKAGE = 0.25

XGB_PARAMS = {
    "objective": "reg:pseudohubererror",
    "n_estimators": 700,
    "learning_rate": 0.030,
    "max_depth": 4,
    "min_child_weight": 100,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.15,
    "reg_lambda": 7.0,
    "gamma": 0.05,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}

DESCRIPTOR_COLUMNS = [
    "known_rows",
    "known_tvt_span",
    "known_md_span",
    "known_gr_mean_last300",
    "known_gr_std_last300",
    "known_gr_slope_last300",
    "xy_span_known",
    "z_drop_known",
    "typewell_corr_last300",
    "typewell_mae_last300",
    "typewell_best_lag_corr",
    "typewell_best_lag_tvt",
    "typewell_best_lag_mae",
    "selected_analog_distance",
]

ROW_FEATURE_COLUMNS = [
    "row_idx_original",
    "MD",
    "X",
    "Y",
    "Z",
    "GR",
    "last_known_tvt_feature",
    "last_known_md",
    "md_from_last_known",
    "md_from_start",
    "eval_progress",
    "dx_from_last",
    "dy_from_last",
    "dz_from_last",
    "horizontal_from_last",
    "distance_from_last",
    "distance_log1p",
]

TYPEWELL_FEATURE_COLUMNS = [
    "tvt_linear_proxy",
    "typewell_gr_linear_tvt",
    "typewell_gr_lag_best",
    "gr_minus_typewell_linear",
    "gr_minus_typewell_bestlag",
]

FEATURE_COLUMNS = ROW_FEATURE_COLUMNS + DESCRIPTOR_COLUMNS + TYPEWELL_FEATURE_COLUMNS
RESIDUAL_FEATURE_COLUMNS = [
    "eval_progress",
    "md_from_last_known",
    "md_from_start",
    "distance_log1p",
    "GR",
    "last_known_tvt_feature",
    "known_tvt_span",
    "known_md_span",
    "known_gr_mean_last300",
    "known_gr_std_last300",
    "known_gr_slope_last300",
    "typewell_corr_last300",
    "typewell_mae_last300",
    "typewell_best_lag_corr",
    "typewell_best_lag_tvt",
    "typewell_best_lag_mae",
    "selected_analog_distance",
    "tvt_linear_proxy",
    "typewell_gr_linear_tvt",
    "typewell_gr_lag_best",
    "gr_minus_typewell_linear",
    "gr_minus_typewell_bestlag",
]


def find_competition_base():
    if KAGGLE_INPUT_ROOT.exists():
        candidates = []
        for path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv"):
            base = path.parent
            if (base / "train").exists() and (base / "test").exists():
                candidates.append(base)
        if candidates:
            return sorted(candidates, key=lambda item: len(str(item)))[0]
    for root in [Path.cwd(), *Path.cwd().parents]:
        base = root / "data" / "raw"
        if (base / "train").exists() and (base / "test").exists() and (base / "sample_submission.csv").exists():
            return base
    return Path("data/raw")

BASE_DIR = find_competition_base()
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
SAMPLE_PATH = BASE_DIR / "sample_submission.csv"
OUTPUT_PATH = KAGGLE_OUTPUT / "submission.csv" if KAGGLE_ROOT.exists() else Path("artifacts/reports/submission__xgboost_residual_stack__2026-06-28.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)

BASE_DIR: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw
OUTPUT_PATH: artifacts\reports\submission__xgboost_residual_stack__2026-06-28.csv


## 2. Helpers

In [2]:
"Data loading, descriptor construction, and inference-safe feature helpers."


def discover_wells(directory):
    return sorted(path.name.replace("__horizontal_well.csv", "") for path in Path(directory).glob("*__horizontal_well.csv"))


def stable_fold(wellname, n_folds):
    digest = hashlib.md5(str(wellname).encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % int(n_folds)


def load_horizontal(path):
    df = pd.read_csv(path)
    df.index.name = "row_idx"
    df["row_idx"] = df.index.astype(int)
    return df


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def load_sample_submission(path):
    sample = pd.read_csv(path)
    split = sample["id"].str.rsplit("_", n=1, expand=True)
    sample["wellname"] = split[0]
    sample["row_idx"] = split[1].astype(int)
    return sample


def safe_corr(left, right):
    a = np.asarray(left, dtype=float)
    b = np.asarray(right, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if int(mask.sum()) < 5:
        return np.nan
    if float(np.nanstd(a[mask])) < 1e-9 or float(np.nanstd(b[mask])) < 1e-9:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


def linear_slope(x_values, y_values):
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    return float(np.polyfit(x[mask], y[mask], 1)[0])


def interp_typewell_gr(typewell, tvt_values, lag=0.0):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float) + lag, clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_descriptor(wellname, directory):
    horizontal = load_horizontal(Path(directory) / f"{wellname}__horizontal_well.csv")
    typewell = load_typewell(directory, wellname)
    known = horizontal.loc[horizontal["TVT_input"].notna()].copy()
    if known.empty:
        known = horizontal.head(1).copy()
    gr = known["GR"].astype(float).clip(0, 300)
    last300 = known.tail(300).copy()
    gr_last300 = last300["GR"].astype(float).clip(0, 300)
    known_tvt = known["TVT_input"].astype(float) if "TVT_input" in known else known.get("TVT", pd.Series(dtype=float)).astype(float)
    last_known_tvt = float(known_tvt.dropna().iloc[-1]) if known_tvt.notna().any() else np.nan
    desc = {
        "wellname": wellname,
        "known_rows": float(len(known)),
        "known_tvt_span": float(np.nanmax(known_tvt) - np.nanmin(known_tvt)) if known_tvt.notna().sum() else np.nan,
        "known_md_span": float(known["MD"].max() - known["MD"].min()) if "MD" in known else np.nan,
        "known_gr_mean_last300": float(gr_last300.mean()) if gr_last300.notna().any() else np.nan,
        "known_gr_std_last300": float(gr_last300.std(ddof=0)) if gr_last300.notna().any() else np.nan,
        "known_gr_slope_last300": linear_slope(last300["MD"], gr_last300) if len(last300) else np.nan,
        "xy_span_known": float(np.sqrt((known["X"].iloc[-1] - known["X"].iloc[0]) ** 2 + (known["Y"].iloc[-1] - known["Y"].iloc[0]) ** 2)) if {"X", "Y"}.issubset(known.columns) and len(known) else np.nan,
        "z_drop_known": float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if "Z" in known and len(known) else np.nan,
        "last_known_tvt": last_known_tvt,
    }
    if len(last300) and np.isfinite(last_known_tvt):
        type_gr = interp_typewell_gr(typewell, last300["TVT_input"].astype(float), lag=0.0)
        desc["typewell_corr_last300"] = safe_corr(gr_last300, type_gr)
        desc["typewell_mae_last300"] = float(np.nanmean(np.abs(gr_last300.to_numpy(float) - type_gr))) if np.isfinite(type_gr).any() else np.nan
        best = {"corr": -np.inf, "lag": np.nan, "mae": np.nan}
        for lag in ANALOG_LAGS:
            shifted = interp_typewell_gr(typewell, last300["TVT_input"].astype(float), lag=lag)
            corr = safe_corr(gr_last300, shifted)
            if np.isfinite(corr) and corr > best["corr"]:
                mae = float(np.nanmean(np.abs(gr_last300.to_numpy(float) - shifted))) if np.isfinite(shifted).any() else np.nan
                best = {"corr": corr, "lag": float(lag), "mae": mae}
        desc["typewell_best_lag_corr"] = best["corr"] if np.isfinite(best["corr"]) else np.nan
        desc["typewell_best_lag_tvt"] = best["lag"]
        desc["typewell_best_lag_mae"] = best["mae"]
    else:
        desc["typewell_corr_last300"] = np.nan
        desc["typewell_mae_last300"] = np.nan
        desc["typewell_best_lag_corr"] = np.nan
        desc["typewell_best_lag_tvt"] = np.nan
        desc["typewell_best_lag_mae"] = np.nan
    return desc


def add_nearest_analog_distance(train_desc, test_desc=None):
    train = train_desc.copy().reset_index(drop=True)
    test = None if test_desc is None else test_desc.copy().reset_index(drop=True)
    feature_cols = [col for col in DESCRIPTOR_COLUMNS if col != "selected_analog_distance" and col in train.columns]
    usable_cols = []
    for col in feature_cols:
        train[col] = pd.to_numeric(train[col], errors="coerce")
        if test is not None and col in test.columns:
            test[col] = pd.to_numeric(test[col], errors="coerce")
        if train[col].notna().any():
            usable_cols.append(col)
    if len(train) < 2 or not usable_cols:
        train["selected_analog_distance"] = 0.0
        if test is not None:
            test["selected_analog_distance"] = 0.0
        return train, test
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    train_x = pipe.fit_transform(train[usable_cols])
    train_dist = pairwise_distances(train_x, train_x)
    np.fill_diagonal(train_dist, np.inf)
    train["selected_analog_distance"] = np.nanmin(train_dist, axis=1)
    if test is not None and not test.empty:
        for col in usable_cols:
            if col not in test.columns:
                test[col] = np.nan
        test_x = pipe.transform(test[usable_cols])
        dist = pairwise_distances(test_x, train_x)
        test["selected_analog_distance"] = np.nanmin(dist, axis=1)
        return train, test
    return train, test


def sample_target_rows(target, max_rows):
    if max_rows is None or len(target) <= max_rows:
        return target.copy()
    idx = np.linspace(0, len(target) - 1, max_rows).round().astype(int)
    return target.iloc[np.unique(idx)].copy()


def build_feature_rows(wellname, directory, descriptor_row, row_indices=None, max_target_rows=None, require_target=True):
    df = load_horizontal(Path(directory) / f"{wellname}__horizontal_well.csv")
    df = df.sort_values("MD").reset_index(drop=True)
    df["row_idx"] = df.index.astype(int)
    known_mask = df["TVT_input"].notna()
    if int(known_mask.sum()) == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    if row_indices is not None:
        target = df.loc[df["row_idx"].isin([int(value) for value in row_indices])].copy()
    else:
        target = df.loc[~known_mask].copy()
    if require_target and "TVT" in target.columns:
        target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    target = sample_target_rows(target, max_target_rows)
    typewell = load_typewell(directory, wellname)
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    gr_values = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)
    out = pd.DataFrame({
        "wellname": wellname,
        "row_idx": target["row_idx"].astype(int).to_numpy(),
        "row_idx_original": target["row_idx"].astype(float).to_numpy(),
        "MD": target["MD"].astype(float).to_numpy(),
        "X": target["X"].astype(float).to_numpy(),
        "Y": target["Y"].astype(float).to_numpy(),
        "Z": target["Z"].astype(float).to_numpy(),
        "GR": gr_values.to_numpy(),
        "last_known_tvt_feature": float(last["TVT_input"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
    })
    if "TVT" in target.columns:
        out["TVT"] = target["TVT"].astype(float).to_numpy()
        out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = np.arange(1, len(out) + 1, dtype=float) / (len(out) + 1e-9)
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(descriptor_row.get("typewell_best_lag_tvt", 0.0)) if pd.notna(descriptor_row.get("typewell_best_lag_tvt", np.nan)) else 0.0
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy(), lag=0.0)
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy(), lag=best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]
    for col in DESCRIPTOR_COLUMNS:
        out[col] = float(descriptor_row[col]) if col in descriptor_row and pd.notna(descriptor_row[col]) else np.nan
    return out


def fit_residual_pipeline(X, y):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=200.0)),
    ])
    pipe.fit(X, y)
    return pipe

## 3. Train Base And Residual Models

In [3]:
"Build training rows, generate OOF base residuals, and fit final base plus Ridge residual stacker."

train_wells = discover_wells(TRAIN_DIR)
test_wells = discover_wells(TEST_DIR)
train_desc = pd.DataFrame([build_descriptor(well, TRAIN_DIR) for well in train_wells])
test_desc = pd.DataFrame([build_descriptor(well, TEST_DIR) for well in test_wells])
train_desc, test_desc = add_nearest_analog_distance(train_desc, test_desc)

train_frames = []
for row in train_desc.to_dict(orient="records"):
    frame = build_feature_rows(
        row["wellname"],
        TRAIN_DIR,
        row,
        row_indices=None,
        max_target_rows=MAX_TARGET_ROWS_PER_WELL,
        require_target=True,
    )
    if not frame.empty:
        train_frames.append(frame)
train_df = pd.concat(train_frames, ignore_index=True)
train_df["oof_fold"] = train_df["wellname"].map(lambda value: stable_fold(value, N_OOF_FOLDS))
X_all = train_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
y_delta = train_df["target_delta"].astype(float)
y_tvt = train_df["TVT"].astype(float)
base_oof_pred = pd.Series(index=train_df.index, dtype=float)

for fold_id in sorted(train_df["oof_fold"].unique()):
    valid_mask = train_df["oof_fold"].eq(fold_id)
    train_mask = ~valid_mask
    fold_model = XGBRegressor(**XGB_PARAMS)
    fold_model.fit(X_all.loc[train_mask], y_delta.loc[train_mask], verbose=False)
    pred_delta = fold_model.predict(X_all.loc[valid_mask])
    base_oof_pred.loc[valid_mask] = train_df.loc[valid_mask, "last_known_tvt_feature"].to_numpy(float) + pred_delta

residual_target = y_tvt - base_oof_pred
residual_model = fit_residual_pipeline(
    train_df[RESIDUAL_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan),
    residual_target.astype(float),
)

base_model = XGBRegressor(**XGB_PARAMS)
base_model.fit(X_all, y_delta, verbose=False)

print("model:", MODEL_NAME)
print("train_wells:", len(train_wells))
print("test_wells:", len(test_wells))
print("train_rows:", len(train_df))
print("n_base_features:", len(FEATURE_COLUMNS))
print("n_residual_features:", len(RESIDUAL_FEATURE_COLUMNS))
print("oof_base_rmse:", float(np.sqrt(np.mean((y_tvt.to_numpy(float) - base_oof_pred.to_numpy(float)) ** 2))))
print("oof_residual_mean:", float(residual_target.mean()))
print("oof_residual_std:", float(residual_target.std(ddof=0)))

model: xgb_ridge_residual_stack_inference_safe
train_wells: 773
test_wells: 3
train_rows: 92760
n_base_features: 36
n_residual_features: 22
oof_base_rmse: 15.03496838024761
oof_residual_mean: 0.36109125065231157
oof_residual_std: 15.030631633559148


## 4. Write Submission

In [4]:
"Generate predictions in sample_submission order and write submission.csv."

sample = load_sample_submission(SAMPLE_PATH)
predictions = {}
test_desc_map = {row["wellname"]: row for row in test_desc.to_dict(orient="records")}

for wellname, group in sample.groupby("wellname", sort=True):
    descriptor_row = test_desc_map.get(wellname)
    if descriptor_row is None:
        descriptor_row = build_descriptor(wellname, TEST_DIR)
        descriptor_row["selected_analog_distance"] = float(np.nanmedian(train_desc["selected_analog_distance"]))
    frame = build_feature_rows(
        wellname,
        TEST_DIR,
        descriptor_row,
        row_indices=group["row_idx"].tolist(),
        max_target_rows=None,
        require_target=False,
    )
    if frame.empty:
        continue
    X_test = frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    pred_delta = base_model.predict(X_test)
    base_pred_tvt = frame["last_known_tvt_feature"].to_numpy(float) + pred_delta
    residual_correction = RESIDUAL_SHRINKAGE * residual_model.predict(frame[RESIDUAL_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan))
    pred_tvt = base_pred_tvt + residual_correction
    for row_idx, value in zip(frame["row_idx"].astype(int), pred_tvt):
        predictions[f"{wellname}_{row_idx}"] = float(value)

submission = sample[["id"]].copy()
submission["tvt"] = submission["id"].map(predictions)

assert list(submission.columns) == ["id", "tvt"]
assert len(submission) == len(sample)
assert submission["id"].equals(sample["id"])
assert submission["tvt"].notna().all()
assert np.isfinite(submission["tvt"].to_numpy()).all()

submission.to_csv(OUTPUT_PATH, index=False)

print("submission_rows:", len(submission))
print("submission_path:", OUTPUT_PATH)
print("prediction_min:", float(submission["tvt"].min()))
print("prediction_max:", float(submission["tvt"].max()))
submission.head()

submission_rows: 14151
submission_path: artifacts\reports\submission__xgboost_residual_stack__2026-06-28.csv
prediction_min: 11597.11480144852
prediction_max: 12222.397576513196


,id,tvt
0,000d7d20_1442,11744.558733
1,000d7d20_1443,11744.602774
2,000d7d20_1444,11744.634488
3,000d7d20_1445,11744.657676
4,000d7d20_1446,11744.676631
